In [11]:
import pandas as pd
import numpy as np
from collections import defaultdict
import networkx as nx
from collections import Counter

In [12]:
# load the clustered SV regions file
clustered_sv = pd.read_csv('../../outputs/leopard_cohort/all_samples_clustering_regions_v2.txt', sep='\t')
clustered_sv = clustered_sv[['chr', 'start.bp', 'end.bp', 'length.bp', 'number.bps', 'sample']]
clustered_sv['chr'] = clustered_sv['chr'].astype(str).replace({'23': 'X', '24': 'Y'})

clustered_sv['cluster_id'] = clustered_sv.apply(
    lambda r: f"{r['sample']}_{r['chr']}_{int(r['start.bp'])}_{int(r['end.bp'])}", axis=1
)
clustered_sv.head()

,chr,start.bp,end.bp,length.bp,number.bps,sample,cluster_id
0,8,102616771,113822490,11205719,42,18138-010BL,18138-010BL_8_102616771_113822490
1,8,126951073,142778150,15827077,84,18138-010BL,18138-010BL_8_126951073_142778150
2,19,11464124,41906615,30442491,13,18138-033BL,18138-033BL_19_11464124_41906615
3,8,91670976,94865219,3194243,10,18138-072PR,18138-072PR_8_91670976_94865219
4,8,114057612,128370696,14313084,13,18138-072PR,18138-072PR_8_114057612_128370696


In [13]:
# load all SVs file
svs = pd.read_csv('../../outputs/leopard_cohort/all_samples_annotated_v2_bedpe.txt', sep='\t')
print(f"Total samples with SVs: {svs['sample'].nunique()}")
svs.head()

Total samples with SVs: 37


,chrom1,start1,end1,chrom2,start2,end2,sample,svclass,length,id,is.clustered,bkdist,catalogue.label
0,1,46213358,46213359,2,159266489,159266490,18138-001BL,translocation,NaN,1,False,113053131,non-clustered_trans
1,1,89010244,89010245,1,89012941,89012942,18138-001BL,deletion,2697.0,2,False,2697,non-clustered_del_1-10Kb
2,1,109304168,109304169,7,24551505,24551506,18138-001BL,translocation,NaN,3,False,84752663,non-clustered_trans
3,1,153700670,153700671,1,153721394,153721395,18138-001BL,tandem-duplication,20724.0,4,False,20724,non-clustered_tds_10-100Kb
4,2,41746061,41746062,12,425300,425301,18138-001BL,translocation,NaN,5,False,41320761,non-clustered_trans


In [14]:
# 1. If linked events are on the same chromosome, merge these clusters and adjust coordinates
LINK_WINDOW_BP = 10_000
MIN_LINKS      = 2

def parse_cluster(cid):
    parts = cid.rsplit('_', 3)
    return parts[0], parts[1], int(parts[2]), int(parts[3])  # sample, chr, start, end

# (sample, chr) → [(start, end, id), ...]
def build_index(id_list):
    idx = defaultdict(list)
    for cid in id_list:
        sample, chrom, start, end = parse_cluster(cid)
        idx[(sample, chrom)].append((start, end, cid))
    return idx

def nearby(idx, sample, chrom, pos, window=LINK_WINDOW_BP):
    return [
        cid for (start, end, cid) in idx.get((sample, chrom), [])
        if (pos - window) <= end and (pos + window) >= start
    ]

# index the original clusters
cluster_idx = build_index(clustered_sv['cluster_id'])

# loop through SVs, collect intra-chr edges only
G = nx.Graph()
for cid in clustered_sv['cluster_id']:
    G.add_node(cid)

pair_support = Counter()

for _, sv in svs.iterrows():
    sample = sv['sample']
    if pd.isna(sample) or str(sv['chrom1']) != str(sv['chrom2']):
        continue      # skip inter-chr SVs

    hits1 = nearby(cluster_idx, sample, str(sv['chrom1']), sv['start1'])
    hits2 = nearby(cluster_idx, sample, str(sv['chrom2']), sv['start2'])

    # collect this SV's pairs as a set so one SV counts at most once per pair
    sv_pairs = {
        tuple(sorted((c1, c2)))
        for c1 in hits1
        for c2 in hits2
        if c1 != c2
    }
    for pair in sv_pairs:
        pair_support[pair] += 1
    
# add edges to graph 
for (c1, c2), n in pair_support.items():
    if n >= MIN_LINKS:
        G.add_edge(c1, c2)

# merge connected components
merged_events = []
for component in nx.connected_components(G):
    parsed    = [parse_cluster(c) for c in component]
    sample    = parsed[0][0]
    chrom     = parsed[0][1]
    mrg_start = min(p[2] for p in parsed)
    mrg_end   = max(p[3] for p in parsed)
    event_id  = f"{sample}_{chrom}_{mrg_start}_{mrg_end}"

    merged_events.append({
        'sample':          sample,
        'chr':             chrom,
        'start':           mrg_start,
        'end':             mrg_end,
        'event_id':        event_id,
        'source_clusters': sorted(component),
        'n_merged':        len(component),
    })

merged_df = (
    pd.DataFrame(merged_events)
    .sort_values(['sample', 'chr', 'start'])
    .reset_index(drop=True)
)

merged_df.to_csv('../../outputs/leopard_cohort/merged_clustered_sv_events.csv', index=False)
print(f"Merged events: {len(merged_df)}  (from {len(clustered_sv)} original clusters)")
merged_df[merged_df['n_merged'] > 1]

Merged events: 13  (from 16 original clusters)


,sample,chr,start,end,event_id,source_clusters,n_merged
0,18138-010BL,8,102616771,142778150,18138-010BL_8_102616771_142778150,"[18138-010BL_8_102616771_113822490, 18138-010B...",2
1,18138-010C1,8,107344701,142778150,18138-010C1_8_107344701_142778150,"[18138-010C1_8_107344701_113830497, 18138-010C...",3


In [16]:
merged_df

,sample,chr,start,end,event_id,source_clusters,n_merged
0,18138-010BL,8,102616771,142778150,18138-010BL_8_102616771_142778150,"[18138-010BL_8_102616771_113822490, 18138-010B...",2
1,18138-010C1,8,107344701,142778150,18138-010C1_8_107344701_142778150,"[18138-010C1_8_107344701_113830497, 18138-010C...",3
2,18138-033BL,19,11464124,41906615,18138-033BL_19_11464124_41906615,[18138-033BL_19_11464124_41906615],1
3,18138-072PR,17,44708123,53197486,18138-072PR_17_44708123_53197486,[18138-072PR_17_44708123_53197486],1
4,18138-072PR,8,91670976,94865219,18138-072PR_8_91670976_94865219,[18138-072PR_8_91670976_94865219],1
5,18138-072PR,8,114057612,128370696,18138-072PR_8_114057612_128370696,[18138-072PR_8_114057612_128370696],1
6,18138-094-F,8,98590450,103238740,18138-094-F_8_98590450_103238740,[18138-094-F_8_98590450_103238740],1
7,18138-094BL,8,94153806,99108746,18138-094BL_8_94153806_99108746,[18138-094BL_8_94153806_99108746],1
8,18138-098BL,17,15256299,18364384,18138-098BL_17_15256299_18364384,[18138-098BL_17_15256299_18364384],1
9,18138-117BL,12,57682154,74315667,18138-117BL_12_57682154_74315667,[18138-117BL_12_57682154_74315667],1
